# Preprocessing — Credit Card Transaction Fraud

Builds the modelling-ready feature files from the raw Sparkov data.

Input (gitignored, see `eda.ipynb`): `data/fraudTrain.csv`, `data/fraudTest.csv`

Output: `data/processed/fraudTrain_features.csv`, `data/processed/fraudTest_features.csv`

This notebook covers:
1. Loading data & dropping rows with missing gender
2. Train-only thresholds
3. Time features (incl. cyclical encodings)
4. Amount and age features
5. Distance features
6. Merchant category
7. Dropping identifiers and redundant columns
8. One-hot encoding (avoiding the dummy variable trap)
9. Multicollinearity check
10. Saving the output

All transformations are target-independent: `is_fraud` is never used to build a feature. It is carried through unchanged in both output files as the label.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "processed"

## 1. Load data

`gender`, `category` and `state` have no missing values in either file. Rows with a missing `gender` are still dropped as a safeguard: with `drop_first=True` encoding (section 8) there is no missing-value indicator, so a missing gender would otherwise be silently encoded as the baseline (`F`).

In [2]:
train = pd.read_csv(DATA_DIR / "fraudTrain.csv", index_col=0)
test = pd.read_csv(DATA_DIR / "fraudTest.csv", index_col=0)

for name, df in [("train", train), ("test", test)]:
    print(f"{name}: {df.shape}, missing gender: {df['gender'].isna().sum():,}")

train = train.dropna(subset=["gender"])
test = test.dropna(subset=["gender"])

train: (1296675, 22), missing gender: 0
test: (555719, 22), missing gender: 0


## 2. Train-only thresholds

`amt_is_large` and `distance_is_large` flag the top 5% of transactions. The 95th-percentile cut-offs are computed on the **training set only** and applied unchanged to the test set, so the test distribution never influences preprocessing.

In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two latitude/longitude pairs."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * 6371 * np.arcsin(np.sqrt(a))


amount_95 = train["amt"].quantile(0.95)
distance_95 = haversine_km(
    train["lat"], train["long"], train["merch_lat"], train["merch_long"]
).quantile(0.95)

print(f"95th-percentile amount:   {amount_95:.2f}")
print(f"95th-percentile distance: {distance_95:.2f} km")

95th-percentile amount:   196.31
95th-percentile distance: 120.50 km


## 3. Time features

The raw timestamp is split into `month`, `day_of_month`, `hour` and `day_of_week` (Monday = 0), plus two flags: `is_weekend` and `is_night` (00:00–05:59).

**No `year` feature:** it has only two values (2019, 2020) and the test set is entirely 2020, so the model cannot learn anything about year that transfers to the test period.

**Cyclical encodings.** Hour, day of week and month wrap around: 23:00 is next to 00:00, Sunday next to Monday, December next to January. As raw integers these neighbours look far apart (23 vs 0). Mapping each value to a point on a circle fixes this:

```
angle = 2π × value / period      (period = 24, 7 or 12)
x_sin = sin(angle),  x_cos = cos(angle)
```

Both are needed: `sin` alone gives the same value for mirror-image points (e.g. 02:00 and 10:00), but the `(sin, cos)` pair is unique for every value.

In [4]:
def add_time_features(df):
    df = df.copy()
    timestamp = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")

    df["month"] = timestamp.dt.month
    df["day_of_month"] = timestamp.dt.day
    df["hour"] = timestamp.dt.hour
    df["day_of_week"] = timestamp.dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype("int8")
    df["is_night"] = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

## 4. Amount and age features

- `log_amt` = `log1p(amt)`, reducing the heavy right skew of `amt`
- `amt_is_large` = 1 when `amt` ≥ the training 95th percentile
- `age` = cardholder age at transaction time, from `dob`

In [5]:
def add_amount_age_features(df, amount_95):
    df = df.copy()
    timestamp = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    dob = pd.to_datetime(df["dob"], errors="coerce")

    df["age"] = ((timestamp - dob).dt.days / 365.25).round().astype("float")
    df["log_amt"] = np.log1p(df["amt"].clip(lower=0))
    df["amt_is_large"] = (df["amt"] >= amount_95).astype("int8")
    return df

## 5. Distance features

Distance between cardholder and merchant (Haversine) is kept only as `log_distance_km` and `distance_is_large`.

Raw `distance_km` and the earlier interaction features (`online_far_transaction` = online × distance, `amount_distance_interaction` = amount × distance) are **not** kept:
- they had VIFs of 11–81, being built from `log_distance_km` and `log_amt`;
- distance shows no relationship with fraud in this dataset (table below). The simulator places every merchant within ~150 km of the cardholder, so no transaction is genuinely "far away".

In [6]:
check = train[["category", "is_fraud"]].copy()
check["distance_km"] = haversine_km(
    train["lat"], train["long"], train["merch_lat"], train["merch_long"]
)
check["channel"] = np.select(
    [check["category"].str.endswith("_net"), check["category"].str.endswith("_pos")],
    ["online", "pos"],
    default="other",
)
check["distance_quintile"] = pd.qcut(
    check["distance_km"], 5, labels=["q1 near", "q2", "q3", "q4", "q5 far"]
)

# Fraud rate (%) is flat across distance for every channel
(
    check.pivot_table(
        index="distance_quintile", columns="channel", values="is_fraud",
        aggfunc="mean", observed=True,
    ) * 100
).round(2)

channel,online,other,pos
distance_quintile,,,
q1 near,1.36,0.23,0.85
q2,1.35,0.27,0.91
q3,1.31,0.24,0.91
q4,1.30,0.25,0.89
q5 far,1.38,0.25,0.86


In [7]:
def add_distance_features(df, distance_95):
    df = df.copy()
    distance_km = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])
    df["log_distance_km"] = np.log1p(distance_km.clip(lower=0))
    df["distance_is_large"] = (distance_km >= distance_95).astype("int8")
    return df

## 6. Merchant category

`category` is kept and one-hot encoded (section 8), so models can tell apart e.g. `grocery_net` and `grocery_pos`.

`is_online`, `is_pos` and a suffix-free `category_base` are **not** created, because each is an exact sum of `category` dummies, which is perfect multicollinearity:
- `is_online` = `category_grocery_net` + `category_misc_net` + `category_shopping_net`
- `is_pos` = the sum of the three `_pos` dummies
- `category_base_shopping` = `category_shopping_net` + `category_shopping_pos`, and so on

Even `category_base` + `is_online` + `is_pos` together are collinear (`is_online + is_pos` = the grocery + misc + shopping base dummies), so the cleanest option is to keep `category` alone.

## 7. Drop identifiers and redundant columns

| Column(s) | Reason |
|---|---|
| `trans_date_trans_time`, `dob` | Replaced by time features and `age` |
| `trans_num`, `cc_num` | Identifiers; a card number has no numeric meaning |
| `first`, `last`, `street` | Personal information |
| `city`, `zip`, `merchant`, `job` | High-cardinality; excluded from this baseline |
| `unix_time` | Same information as `month`/`day_of_month`/`hour` (VIF > 100,000) |
| `merch_lat`, `merch_long` | Near-copies of cardholder coordinates (VIF 78–570); their only distinct information is the distance |
| `lat`, `long` | Largely determined by `state` (VIF up to 650 with the state dummies); `state` is kept as the more interpretable location feature |

In [8]:
COLUMNS_TO_DROP = [
    "trans_date_trans_time", "dob", "trans_num", "cc_num",
    "first", "last", "street", "city", "zip", "merchant", "job",
    "unix_time", "merch_lat", "merch_long", "lat", "long",
]


def build_features(df):
    df = add_time_features(df)
    df = add_amount_age_features(df, amount_95)
    df = add_distance_features(df, distance_95)
    return df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])


train_features = build_features(train)
test_features = build_features(test)
train_features.head()

,category,amt,gender,state,city_pop,is_fraud,month,day_of_month,hour,day_of_week,is_weekend,is_night,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,age,log_amt,amt_is_large,log_distance_km,distance_is_large
0,misc_net,4.97,F,NC,3495,0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,31.0,1.786747,0,4.376984,0
1,grocery_pos,107.23,F,WA,149,0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,41.0,4.684259,0,3.440808,0
2,entertainment,220.11,M,ID,4154,0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,57.0,5.398660,1,4.693237,0
3,gas_transport,45.00,M,MT,1939,0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,52.0,3.828641,0,4.571337,0
4,misc_pos,41.96,M,VA,99,0,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,33.0,3.760269,0,4.363821,0


## 8. One-hot encoding

The remaining text columns (`category`, `gender`, `state`) become 0/1 indicator columns. Numbering categories instead (grocery = 0, shopping = 1, …) would invent an order and distances that don't exist.

**Dummy variable trap.** A full set of dummies always sums to 1 (e.g. `gender_F + gender_M = 1`), which is perfectly collinear with the intercept. `drop_first=True` drops one category per column as the **baseline**; each remaining coefficient is read relative to it.

The baseline is the **most frequent category in the training set**: `F` for gender, `gas_transport` for category, `TX` for state. A rare baseline causes near-collinearity: with `AK` (2,120 rows) as the baseline, the other 50 state dummies sum to 1 for 99.8% of rows, which pushes state VIFs up to ~40.

`is_fraud` is set aside before encoding and added back afterwards. Train and test are encoded against the same category list so both files end up with identical columns. Indicators are stored sparsely to save memory.

In [9]:
def one_hot_encode(train_features, test_features):
    train_target = train_features["is_fraud"]
    test_target = test_features["is_fraud"]
    train_x = train_features.drop(columns=["is_fraud"])
    test_x = test_features.drop(columns=["is_fraud"])

    categorical_columns = train_x.select_dtypes(include=["object", "category"]).columns.tolist()
    print("Encoding (n categories, baseline):",
          {c: (train_x[c].nunique(), train_x[c].value_counts().index[0]) for c in categorical_columns})

    train_parts = [train_x.drop(columns=categorical_columns)]
    test_parts = [test_x.drop(columns=categorical_columns)]
    for column in categorical_columns:
        # Most frequent training category first, so drop_first makes it the baseline
        baseline = train_x[column].value_counts().index[0]
        others = sorted(
            set(pd.concat([train_x[column], test_x[column]]).dropna().unique()) - {baseline}
        )
        categories = pd.Index([baseline] + others)
        dtype = pd.CategoricalDtype(categories=categories)
        train_parts.append(pd.get_dummies(train_x[column].astype(dtype), prefix=column, drop_first=True, sparse=True))
        test_parts.append(pd.get_dummies(test_x[column].astype(dtype), prefix=column, drop_first=True, sparse=True))

    train_encoded = pd.concat(train_parts, axis=1).reset_index(drop=True)
    test_encoded = pd.concat(test_parts, axis=1).reset_index(drop=True)
    train_encoded.insert(0, "is_fraud", train_target.reset_index(drop=True).astype("int8"))
    test_encoded.insert(0, "is_fraud", test_target.reset_index(drop=True).astype("int8"))
    return train_encoded, test_encoded


train_encoded, test_encoded = one_hot_encode(train_features, test_features)

print("train:", train_encoded.shape, "| test:", test_encoded.shape)
print("Same columns:", list(train_encoded.columns) == list(test_encoded.columns))

Encoding (n categories, baseline): {'category': (14, 'gas_transport'), 'gender': (2, 'F'), 'state': (51, 'TX')}
train: (1296675, 84) | test: (555719, 84)
Same columns: True


### Final dataset preview

All columns of the processed training data (`is_fraud` first, then numeric features, then the one-hot indicators).

In [10]:
train_encoded.head()

,is_fraud,amt,city_pop,month,day_of_month,hour,day_of_week,is_weekend,is_night,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,age,log_amt,amt_is_large,log_distance_km,distance_is_large,category_entertainment,category_food_dining,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_M,state_AK,state_AL,state_AR,state_AZ,state_CA,state_CO,state_CT,state_DC,state_DE,state_FL,state_GA,state_HI,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_MT,state_NC,state_ND,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY
0,0,4.97,3495,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,31.0,1.786747,0,4.376984,0,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,0,107.23,149,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,41.0,4.684259,0,3.440808,0,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,0,220.11,4154,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,57.0,5.398660,1,4.693237,0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,0,45.00,1939,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,52.0,3.828641,0,4.571337,0,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,0,41.96,99,1,1,0,1,0,1,0.0,1.0,0.781831,0.62349,0.5,0.866025,33.0,3.760269,0,4.363821,0,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False


In [11]:
columns = pd.Series(train_encoded.columns)
groups = {
    "target": columns[columns == "is_fraud"],
    "category dummies": columns[columns.str.startswith("category_")],
    "gender dummies": columns[columns.str.startswith("gender_")],
    "state dummies": columns[columns.str.startswith("state_")],
}
groups["numeric features"] = columns[~columns.isin(pd.concat(groups.values()))]
for name, cols in groups.items():
    print(f"{name} ({len(cols)}): {', '.join(cols)}\n")

target (1): is_fraud

category dummies (13): category_entertainment, category_food_dining, category_grocery_net, category_grocery_pos, category_health_fitness, category_home, category_kids_pets, category_misc_net, category_misc_pos, category_personal_care, category_shopping_net, category_shopping_pos, category_travel

gender dummies (1): gender_M

state dummies (50): state_AK, state_AL, state_AR, state_AZ, state_CA, state_CO, state_CT, state_DC, state_DE, state_FL, state_GA, state_HI, state_IA, state_ID, state_IL, state_IN, state_KS, state_KY, state_LA, state_MA, state_MD, state_ME, state_MI, state_MN, state_MO, state_MS, state_MT, state_NC, state_ND, state_NE, state_NH, state_NJ, state_NM, state_NV, state_NY, state_OH, state_OK, state_OR, state_PA, state_RI, state_SC, state_SD, state_TN, state_UT, state_VA, state_VT, state_WA, state_WI, state_WV, state_WY

numeric features (19): amt, city_pop, month, day_of_month, hour, day_of_week, is_weekend, is_night, hour_sin, hour_cos, dow_sin, d

## 9. Multicollinearity check

Two checks on a 200,000-row sample of the training features (standardised, plus an intercept column):

- **Rank:** full rank means no exact linear dependency remains (no dummy trap, no redundant derived columns).
- **VIF (variance inflation factor):** `1 / (1 − R²)` from regressing each feature on all others. Above ~10 is usually considered problematic.

All VIFs are below 10. The highest are the time features, which is expected because `hour`/`hour_sin`/`hour_cos`/`is_night` all describe the same hour and `day_of_week`/`dow_*`/`is_weekend` the same day. Multicollinearity only matters for interpreting individual logistic-regression coefficients; tree models are unaffected.

For reference, before this cleanup the VIFs reached ~160,000 (`year`/`unix_time`/`month`), ~650 (`lat`/`long`/`merch_lat`/`merch_long`), ~80 (distance interactions) and ~40 (state dummies with a rare baseline).

In [12]:
sample = train_encoded.drop(columns=["is_fraud"]).sample(200_000, random_state=0)
sample = sample.apply(
    lambda s: s.sparse.to_dense() if isinstance(s.dtype, pd.SparseDtype) else s
).astype(float)
Z = (sample - sample.mean()) / sample.std()
M = np.column_stack([np.ones(len(Z)), Z.to_numpy()])

print(f"Columns (incl. intercept): {M.shape[1]}, rank: {np.linalg.matrix_rank(M)}")


def vif(Z):
    values = {}
    for column in Z.columns:
        y = Z[column].to_numpy()
        X = np.column_stack([np.ones(len(Z)), Z.drop(columns=column).to_numpy()])
        residual = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
        values[column] = y.var() / residual.var()
    return pd.Series(values).sort_values(ascending=False)


vifs = vif(Z)
vifs.head(15).round(1)

Columns (incl. intercept): 84, rank: 84


hour                       9.5
is_weekend                 9.4
day_of_week                8.9
is_night                   7.0
hour_sin                   3.5
dow_sin                    3.0
hour_cos                   3.0
dow_cos                    2.8
category_home              2.6
category_kids_pets         2.5
month                      2.5
month_sin                  2.3
category_personal_care     2.2
category_shopping_pos      2.2
category_health_fitness    2.2
dtype: float64

## 10. Save

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_encoded.to_csv(OUTPUT_DIR / "fraudTrain_features.csv", index=False)
test_encoded.to_csv(OUTPUT_DIR / "fraudTest_features.csv", index=False)

print(f"Wrote {train_encoded.shape[0]:,} rows x {train_encoded.shape[1]:,} columns (train, incl. is_fraud)")
print(f"Wrote {test_encoded.shape[0]:,} rows x {test_encoded.shape[1]:,} columns (test, incl. is_fraud)")

Wrote 1,296,675 rows x 84 columns (train, incl. is_fraud)
Wrote 555,719 rows x 84 columns (test, incl. is_fraud)
